<div align='center'>

# 🤖 Taller de Inteligencia Artificial
## ¿Puede una máquina aprender a ver? 👁️‍🗨️
### Clasificador de Perros 🐶 vs Gatos 🐱

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TU_USUARIO/TU_REPOSITORIO/blob/main/Taller_IA_Clasificador_V02.ipynb)

**Nivel:** Preparatoria &nbsp;|&nbsp; **Duración:** ~60 minutos &nbsp;|&nbsp; **Modalidad:** Hands-on

</div>

---

## 🌟 ¿Qué aprenderás hoy?

Al terminar este taller serás capaz de:

| # | Habilidad |
|---|---|
| ✅ | Entender qué es una **Red Neuronal** y cómo aprende |
| ✅ | Cargar y preparar un **dataset** de imágenes |
| ✅ | Usar **Transfer Learning** (tecnología usada en la industria) |
| ✅ | Entrenar tu propia **IA** desde cero |
| ✅ | Probar tu modelo con **fotos reales** |

---

## 🏭 ¿Por qué importa esto?

La visión por computadora (Computer Vision) es una de las ramas más poderosas de la IA. Hoy se usa en:

- 🏥 **Medicina** → Detectar tumores en radiografías
- 🚗 **Autos autónomos** → Reconocer peatones y señales de tráfico
- 🏭 **Industria** → Inspección de calidad en líneas de producción
- 📱 **Tu cel** → Desbloqueo por reconocimiento facial

Lo que construiremos hoy usa **exactamente la misma tecnología**. 🔥

---
## 🧠 Un poco de teoría (¡prometo que es rápido!)

### ¿Cómo aprende una IA a ver imágenes?

Imagina que cada imagen es una cuadrícula gigante de números (píxeles). Una **Red Neuronal Convolucional (CNN)** aprende a detectar patrones:

```
Imagen → [Detecta bordes] → [Detecta formas] → [Detecta orejas, hocico...] → ¡Es un Perro!
```

### ¿Qué es Transfer Learning?

En lugar de entrenar una IA desde cero (lo cual toma días y mucho poder de cómputo), vamos a usar **MobileNetV2**: un modelo que Google entrenó con **1.2 millones de imágenes** durante semanas.

Nosotros solo le enseñamos la parte final: "distinguir entre perro y gato". ¡Es como contratar a un experto y enseñarle solo lo que necesitas! 🎯

---
# PASO 1 — 📦 Cargando las herramientas

Primero importamos las librerías que usaremos. Piensa en esto como abrir tu caja de herramientas antes de comenzar a construir.

In [ ]:
# ─────────────────────────────────────────────
#  PASO 1: Importar librerías
# ─────────────────────────────────────────────

import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import requests
from io import BytesIO

# Verificamos que TensorFlow detecte GPU (hace el entrenamiento ~10x más rápido)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detectada: {gpus[0].name}")
    print("   El entrenamiento será MUCHO más rápido 🚀")
else:
    print("⚠️  No se detectó GPU. Ve a: Entorno de ejecución → Cambiar tipo de entorno → T4 GPU")

print(f"\n📦 TensorFlow versión: {tf.__version__}")
print("✅ ¡Todas las herramientas listas!")

---
# PASO 2 — 🐾 Cargando el dataset

Vamos a cargar el dataset **Cats vs Dogs** directamente desde TensorFlow. Contiene **23,262 imágenes** de perros y gatos.

> 💡 **¿Sabías que?** Para que una IA aprenda bien, necesita miles de ejemplos. Es como cuando tú aprendiste a leer: tuviste que ver las letras *muchas* veces antes de reconocerlas automáticamente.

In [ ]:
# ─────────────────────────────────────────────
#  PASO 2A: Cargar el dataset
# ─────────────────────────────────────────────

IMG_SIZE = 160      # Tamaño de imagen (160x160 píxeles)
BATCH_SIZE = 32     # Cuántas imágenes procesar a la vez
AUTOTUNE = tf.data.AUTOTUNE

print("⏳ Descargando dataset... (puede tardar 1-2 minutos)")

# Cargamos el dataset dividido: 80% entrenamiento, 20% validación
(train_ds_raw, val_ds_raw), info = tfds.load(
    'cats_vs_dogs',
    split=['train[:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True
)

clases = info.features['label'].names
n_clases = info.features['label'].num_classes
n_train = int(info.splits['train'].num_examples * 0.8)
n_val   = int(info.splits['train'].num_examples * 0.2)

print(f"\n📊 Dataset cargado exitosamente:")
print(f"   🐱 Clases: {clases}")
print(f"   🏋️  Imágenes de entrenamiento: {n_train:,}")
print(f"   🧪 Imágenes de validación:    {n_val:,}")
print(f"   📐 Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE} píxeles")

In [ ]:
# ─────────────────────────────────────────────
#  PASO 2B: Visualizar algunas imágenes
# ─────────────────────────────────────────────

print("🖼️  Ejemplos del dataset:")
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('Muestra del Dataset — Cats vs Dogs', fontsize=14, fontweight='bold')

nombres_clases = ['🐱 Gato', '🐶 Perro']

for i, (imagen, etiqueta) in enumerate(train_ds_raw.take(10)):
    ax = axes[i // 5][i % 5]
    ax.imshow(imagen)
    ax.set_title(nombres_clases[etiqueta.numpy()], fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()
print("\n💡 Cada imagen tiene una ETIQUETA (gato o perro).")
print("   La IA aprende a asociar patrones visuales con estas etiquetas.")

---
# PASO 3 — ⚙️ Preparando los datos

Las imágenes del dataset vienen en diferentes tamaños. Necesitamos estandarizarlas antes de entrenar.

También aplicaremos **Data Augmentation**: técnica que genera variaciones de las imágenes (voltear, zoom, etc.) para que la IA sea más robusta. ¡Es como practicar con distintos tipos de exámenes!

In [ ]:
# ─────────────────────────────────────────────
#  PASO 3A: Funciones de preprocesamiento
# ─────────────────────────────────────────────

def preprocesar(imagen, etiqueta):
    """Redimensiona y normaliza la imagen."""
    imagen = tf.image.resize(imagen, (IMG_SIZE, IMG_SIZE))
    imagen = tf.cast(imagen, tf.float32) / 255.0  # Valores entre 0 y 1
    return imagen, etiqueta

# Capa de Data Augmentation (solo para entrenamiento)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),    # Voltear horizontalmente
    tf.keras.layers.RandomRotation(0.1),          # Rotar un poco
    tf.keras.layers.RandomZoom(0.1),              # Zoom aleatorio
], name='data_augmentation')

def preprocesar_train(imagen, etiqueta):
    imagen, etiqueta = preprocesar(imagen, etiqueta)
    imagen = data_augmentation(imagen, training=True)
    return imagen, etiqueta

# Construir pipelines de datos optimizados
train_ds = (
    train_ds_raw
    .map(preprocesar_train, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

val_ds = (
    val_ds_raw
    .map(preprocesar, num_parallel_calls=AUTOTUNE)
    .cache()
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print("✅ Pipeline de datos listo")
print(f"   Tamaño de imagen: {IMG_SIZE}x{IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print("   Data Augmentation: Flip, Rotación, Zoom")

In [ ]:
# ─────────────────────────────────────────────
#  PASO 3B: Ver el efecto del Data Augmentation
# ─────────────────────────────────────────────

print("🔄 Así se ven las imágenes con Data Augmentation:")
imagen_ejemplo, etiqueta_ejemplo = next(iter(train_ds_raw.take(1)))
imagen_ejemplo = tf.image.resize(imagen_ejemplo, (IMG_SIZE, IMG_SIZE)) / 255.0

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
axes[0].imshow(imagen_ejemplo)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

for i in range(1, 6):
    img_aug = data_augmentation(tf.expand_dims(imagen_ejemplo, 0), training=True)[0]
    axes[i].imshow(img_aug)
    axes[i].set_title(f'Variante {i}')
    axes[i].axis('off')

plt.suptitle('Data Augmentation: misma imagen, distintas variaciones', fontsize=12)
plt.tight_layout()
plt.show()
print("\n💡 Con estas variaciones, la IA aprende a reconocer perros y gatos")
print("   sin importar ángulo, iluminación o posición.")

---
# PASO 4 — 🧠 Construyendo el modelo con Transfer Learning

Aquí está la magia: usaremos **MobileNetV2**, una red neuronal que Google entrenó con millones de imágenes. Tomaremos sus "conocimientos" y les añadiremos una capa final para clasificar entre perros y gatos.

```
┌────────────────────────────────────────────┐
│         MobileNetV2 (pre-entrenado)        │  ← Ya sabe detectar formas, texturas, etc.
│     (congelado — no lo volvemos a          │
│      entrenar para ahorrar tiempo)         │
└──────────────────────┬─────────────────────┘
                       │
              ┌────────▼────────┐
              │  Nuestra capa  │  ← Esto SÍ entrenamos nosotros
              │   final (nueva) │
              └────────┬────────┘
                       │
               ┌───────▼───────┐
               │  Perro / Gato │
               └───────────────┘
```

In [ ]:
# ─────────────────────────────────────────────
#  PASO 4: Construir el modelo
# ─────────────────────────────────────────────

print("⏳ Cargando MobileNetV2 (pre-entrenado con ImageNet)...")

# Base: MobileNetV2 pre-entrenado
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,      # Sin la capa de clasificación original
    weights='imagenet'      # Pesos pre-entrenados
)

# Congelamos la base (no la reentrenamos → ahorra tiempo)
base_model.trainable = False

# Construimos nuestro modelo completo
modelo = tf.keras.Sequential([
    # Entrada
    tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='imagen_entrada'),

    # Base pre-entrenada
    base_model,

    # Reducción de dimensiones
    tf.keras.layers.GlobalAveragePooling2D(name='pooling'),

    # Regularización para evitar sobreajuste
    tf.keras.layers.Dropout(0.2, name='dropout'),

    # Capa de clasificación final
    # sigmoid → salida entre 0 (gato) y 1 (perro)
    tf.keras.layers.Dense(1, activation='sigmoid', name='clasificacion')
], name='ClasificadorPerroGato')

# Compilar: definimos cómo aprenderá
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("✅ Modelo construido exitosamente")
print(f"\n📋 Resumen del modelo:")
modelo.summary()

In [ ]:
# Contamos parámetros entrenables vs totales
total_params = modelo.count_params()
trainable_params = sum([tf.size(w).numpy() for w in modelo.trainable_weights])
frozen_params = total_params - trainable_params

print(f"📊 Parámetros del modelo:")
print(f"   Total de parámetros:      {total_params:>10,}")
print(f"   Parámetros a entrenar:    {trainable_params:>10,}  ← Solo estos aprenden")
print(f"   Parámetros congelados:    {frozen_params:>10,}  ← Ya vienen entrenados")
print(f"\n   Solo entrenamos el {trainable_params/total_params*100:.1f}% del modelo. ¡Por eso es tan rápido!")

---
# PASO 5 — 🏋️ ¡Entrenando la IA!

Aquí viene la parte más emocionante. El modelo verá miles de imágenes y ajustará sus parámetros internos para mejorar.

**¿Qué significa cada número?**
- `loss` → Qué tan equivocado está (más bajo = mejor)
- `accuracy` → Qué porcentaje acierta (más alto = mejor)
- `val_accuracy` → Precisión con imágenes **que nunca vio** (el número más importante)

In [ ]:
# ─────────────────────────────────────────────
#  PASO 5: Entrenar el modelo
# ─────────────────────────────────────────────

EPOCHS = 5  # Número de veces que verá todo el dataset
            # Con GPU: ~2-3 min. Sin GPU: ~15 min.

print(f"🏋️  Iniciando entrenamiento ({EPOCHS} épocas)...")
print(f"   Cada época = el modelo ve ~{n_train:,} imágenes")
print("   " + "─"*50)

# Callback para guardar el mejor modelo
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    'mejor_modelo.keras',
    save_best_only=True,
    monitor='val_accuracy',
    verbose=0
)

# Callback para detener si no mejora
early_stop_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

historial = modelo.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=[checkpoint_cb, early_stop_cb]
)

print("\n✅ ¡Entrenamiento completado!")

---
# PASO 6 — 📈 Analizando los resultados

Visualicemos cómo evolucionó el aprendizaje época por época.

In [ ]:
# ─────────────────────────────────────────────
#  PASO 6: Gráficas de entrenamiento
# ─────────────────────────────────────────────

acc      = historial.history['accuracy']
val_acc  = historial.history['val_accuracy']
loss     = historial.history['loss']
val_loss = historial.history['val_loss']
epochs_range = range(1, len(acc) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Curvas de Aprendizaje del Modelo', fontsize=14, fontweight='bold')

# Gráfica 1: Precisión
ax1.plot(epochs_range, acc,     'b-o', label='Entrenamiento', linewidth=2)
ax1.plot(epochs_range, val_acc, 'r-o', label='Validación',    linewidth=2)
ax1.set_title('Precisión (Accuracy)', fontsize=12)
ax1.set_xlabel('Época')
ax1.set_ylabel('Precisión')
ax1.legend()
ax1.set_ylim([0, 1])
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax1.grid(True, alpha=0.3)

# Gráfica 2: Pérdida
ax2.plot(epochs_range, loss,     'b-o', label='Entrenamiento', linewidth=2)
ax2.plot(epochs_range, val_loss, 'r-o', label='Validación',    linewidth=2)
ax2.set_title('Pérdida (Loss)', fontsize=12)
ax2.set_xlabel('Época')
ax2.set_ylabel('Pérdida')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

precision_final = max(val_acc)
print(f"\n🏆 Mejor precisión de validación: {precision_final:.1%}")
print(f"   Esto significa que la IA acierta {precision_final:.1%} de las veces")
print(f"   con imágenes que NUNCA había visto antes.")

if precision_final > 0.90:
    print("\n🌟 ¡Excelente! Tu modelo tiene más del 90% de precisión.")
elif precision_final > 0.80:
    print("\n✅ ¡Muy bien! Tu modelo supera el 80% de precisión.")
else:
    print("\n💪 Tu modelo está aprendiendo. Prueba más épocas.")

---
# PASO 7 — 🔍 ¡Probemos el modelo con fotos reales!

Ahora la parte más divertida: vamos a darle imágenes reales a nuestra IA y ver qué dice.

In [ ]:
# ─────────────────────────────────────────────
#  PASO 7A: Función para predecir cualquier imagen
# ─────────────────────────────────────────────

def predecir_imagen(ruta_o_url, mostrar=True):
    """
    Recibe la ruta de una imagen o una URL
    y devuelve la predicción del modelo.
    """
    # Cargar imagen
    if ruta_o_url.startswith('http'):
        response = requests.get(ruta_o_url)
        img = Image.open(BytesIO(response.content)).convert('RGB')
    else:
        img = Image.open(ruta_o_url).convert('RGB')

    # Preprocesar exactamente igual que en entrenamiento
    img_array = np.array(img.resize((IMG_SIZE, IMG_SIZE))) / 255.0
    img_batch = np.expand_dims(img_array, 0)  # Añadir dimensión de batch

    # Predecir
    prediccion = modelo.predict(img_batch, verbose=0)[0][0]

    # Interpretar resultado
    # sigmoid → 0 = Gato, 1 = Perro
    if prediccion > 0.5:
        clase = '🐶 PERRO'
        confianza = prediccion
        color = '#FF6B35'
    else:
        clase = '🐱 GATO'
        confianza = 1 - prediccion
        color = '#6B8CFF'

    if mostrar:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                        gridspec_kw={'width_ratios': [1.5, 1]})

        ax1.imshow(np.array(img))
        ax1.axis('off')
        ax1.set_title('Imagen de entrada', fontsize=11)

        # Barra de confianza
        categorias = ['🐱 Gato', '🐶 Perro']
        valores    = [1 - prediccion, prediccion]
        colores    = ['#6B8CFF', '#FF6B35']
        bars = ax2.barh(categorias, valores, color=colores, height=0.5)
        ax2.set_xlim(0, 1)
        ax2.set_xlabel('Probabilidad')
        ax2.set_title('Resultado del modelo', fontsize=11)
        ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        ax2.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

        for bar, val in zip(bars, valores):
            ax2.text(min(val + 0.02, 0.95), bar.get_y() + bar.get_height()/2,
                    f'{val:.1%}', va='center', fontweight='bold')

        fig.suptitle(f'Predicción: {clase}  |  Confianza: {confianza:.1%}',
                    fontsize=13, fontweight='bold', color=color)
        plt.tight_layout()
        plt.show()

    return clase, confianza

print("✅ Función de predicción lista")
print("   Uso: predecir_imagen('ruta/imagen.jpg')")
print("   O:   predecir_imagen('https://url-de-imagen.jpg')")

In [ ]:
# ─────────────────────────────────────────────
#  PASO 7B: Probar con imágenes de validación
# ─────────────────────────────────────────────

print("🔍 Probando con imágenes del dataset de validación:\n")

nombres_clases = {0: '🐱 Gato (real)', 1: '🐶 Perro (real)'}
aciertos = 0
total = 8

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Predicciones en imágenes de validación', fontsize=13, fontweight='bold')

for i, (imagen, etiqueta) in enumerate(val_ds_raw.take(total)):
    img_resize = tf.image.resize(imagen, (IMG_SIZE, IMG_SIZE)) / 255.0
    pred = modelo.predict(tf.expand_dims(img_resize, 0), verbose=0)[0][0]

    pred_clase = 1 if pred > 0.5 else 0
    etiqueta_num = etiqueta.numpy()
    correcto = pred_clase == etiqueta_num
    if correcto:
        aciertos += 1

    ax = axes[i // 4][i % 4]
    ax.imshow(imagen.numpy())

    confianza = pred if pred > 0.5 else 1 - pred
    emoji = '✅' if correcto else '❌'
    pred_nombre = '🐶 Perro' if pred_clase == 1 else '🐱 Gato'
    real_nombre = nombres_clases[etiqueta_num]

    ax.set_title(f"{emoji} Pred: {pred_nombre}\n({confianza:.0%}) | Real: {real_nombre.split()[1]}",
                fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()
print(f"\n📊 Resultado: {aciertos}/{total} correctos ({aciertos/total:.0%})")

In [ ]:
# ─────────────────────────────────────────────
#  PASO 7C: ¡Sube TU propia imagen!
# ─────────────────────────────────────────────

from google.colab import files

print("📤 ¡Sube una imagen de tu perro o gato!")
print("   Formatos aceptados: .jpg, .jpeg, .png")
print("   (Si no tienes una, pasa al siguiente bloque)")
print()

uploaded = files.upload()

for nombre_archivo in uploaded.keys():
    print(f"\n📁 Analizando: {nombre_archivo}")
    predecir_imagen(nombre_archivo)

In [ ]:
# ─────────────────────────────────────────────
#  PASO 7D: Probar con URLs de internet
#  (Si no tienes imagen propia, usa estas)
# ─────────────────────────────────────────────

# URLs de prueba — puedes cambiarlas por otras
urls_prueba = [
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/4/43/Cute_dog.jpg/320px-Cute_dog.jpg",
     "Perro de Wikipedia"),
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/b/bb/Kittyply_edit1.jpg/320px-Kittyply_edit1.jpg",
     "Gato de Wikipedia"),
]

for url, descripcion in urls_prueba:
    print(f"\n🔍 Analizando: {descripcion}")
    try:
        clase, confianza = predecir_imagen(url)
    except Exception as e:
        print(f"   ⚠️  Error al cargar la imagen: {e}")
        print("   Prueba con otra URL.")

---
# PASO 8 — 🎲 ¡Desafío! ¿Puede confundir a tu IA?

Vamos a probar casos difíciles. ¿Qué pasa cuando la imagen es ambigua o poco común?

In [ ]:
# ─────────────────────────────────────────────
#  PASO 8: Casos desafiantes
# ─────────────────────────────────────────────

print("🎲 Casos interesantes para probar:")
print("   ¿Qué pasa con imágenes que NO son perros ni gatos?")
print("   ¿O imágenes de baja calidad? ¿O dibujos?")
print()

casos_desafiantes = [
    # URL                                                                    Descripción
    ("https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg/320px-Everest_North_Face_toward_Base_Camp_Tibet_Luca_Galuzzi_2006.jpg",
     "Una montaña (ni perro ni gato)"),
]

for url, descripcion in casos_desafiantes:
    print(f"\n🔍 Probando: {descripcion}")
    try:
        clase, confianza = predecir_imagen(url)
        print(f"   La IA dijo: {clase} (confianza: {confianza:.1%})")
        print("   💡 Reflexión: El modelo SIEMPRE debe elegir entre Perro y Gato")
        print("      porque fue entrenado solo para eso. En la industria,")
        print("      se añade una clase 'otro' para casos ambiguos.")
    except Exception as e:
        print(f"   ⚠️  {e}")

---
# PASO 9 — 💾 Guardando el modelo

Un modelo entrenado puede guardarse y reutilizarse. ¡No tienes que volver a entrenarlo cada vez!

In [ ]:
# ─────────────────────────────────────────────
#  PASO 9: Guardar y cargar modelo
# ─────────────────────────────────────────────

# Guardar
modelo.save('clasificador_perro_gato.keras')
print("💾 Modelo guardado como: clasificador_perro_gato.keras")

# Descargar a tu computadora
from google.colab import files
files.download('clasificador_perro_gato.keras')
print("📥 ¡Modelo descargado a tu computadora!")
print()
print("Para cargarlo en otro notebook:")
print("  modelo = tf.keras.models.load_model('clasificador_perro_gato.keras')")

---
# 🎓 ¡Felicidades! Completaste el taller

## ¿Qué construiste hoy?

```
  Imágenes → Preprocesamiento → MobileNetV2 → Clasificación → Perro / Gato
```

Un clasificador de imágenes con:
- Transfer Learning desde un modelo entrenado con 1.2M imágenes
- Data Augmentation para mayor robustez
- ~90%+ de precisión

## 🚀 ¿Qué sigue?

| Nivel | Siguiente paso |
|-------|---------------|
| 🟢 Básico | Experimenta con más épocas o diferentes modelos base |
| 🟡 Intermedio | Añade más clases (razas de perros, tipos de gatos) |
| 🔴 Avanzado | Fine-tuning completo del modelo, despliegue en web |

## 📚 Recursos para seguir aprendiendo

- [TensorFlow tutorials](https://www.tensorflow.org/tutorials)
- [fast.ai](https://www.fast.ai) — Cursos gratuitos de Deep Learning
- [Kaggle](https://www.kaggle.com) — Competencias de IA con datasets reales
- [Google ML Crash Course](https://developers.google.com/machine-learning/crash-course)

---

<div align='center'>

**¿Preguntas? ¡Levanta la mano! 🙋**

</div>

---
## 🔧 BONUS — Fine-Tuning (Para los más aventureros)

¿Ya terminaste antes que los demás? Prueba el **Fine-Tuning**: descongelamos las últimas capas de MobileNetV2 y las entrenamos también, para exprimir más precisión del modelo.

In [ ]:
# ─────────────────────────────────────────────
#  BONUS: Fine-Tuning
# ─────────────────────────────────────────────

print("🔓 Activando Fine-Tuning...")

# Descongelar las últimas 20 capas de MobileNetV2
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

capas_entrenables = sum(1 for l in base_model.layers if l.trainable)
print(f"   Capas descongeladas en la base: {capas_entrenables}")

# Compilar con learning rate MUY bajo (para no destruir los pesos pre-entrenados)
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # 100x más bajo
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("\n🏋️  Ejecutando Fine-Tuning (3 épocas adicionales)...")
historial_ft = modelo.fit(
    train_ds,
    epochs=3,
    validation_data=val_ds
)

precision_ft = max(historial_ft.history['val_accuracy'])
print(f"\n🎯 Precisión con Fine-Tuning: {precision_ft:.1%}")
print(f"   Mejora vs. modelo anterior: +{(precision_ft - max(val_acc)):.1%}")